# Installs and Imports

In [ ]:
!pip install -U bitsandbytes trl peft

import os
import json
import torch
import codecs
import warnings
import pandas as pd
import numpy as np
import dataclasses
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    BitsAndBytesConfig,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
)
from transformers.trainer_utils import get_last_checkpoint
from transformers.trainer_callback import TrainerState
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
warnings.filterwarnings("ignore")

# Settings

In [ ]:
# Patch TrainerState.save_to_json to handle NumPy arrays recursively
def recursive_convert(obj):
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    elif isinstance(obj, dict):
        return {k: recursive_convert(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [recursive_convert(v) for v in obj]
    else:
        return obj

def custom_save_to_json(self, json_path: str):
    state_dict = dataclasses.asdict(self)
    state_dict = recursive_convert(state_dict)
    with codecs.open(json_path, 'w', encoding='utf-8') as f:
        json.dump(state_dict, f, separators=(',', ':'), sort_keys=True, indent=4)

TrainerState.save_to_json = custom_save_to_json
model_name_or_path = "/kaggle/input/llama-3.1/transformers/8b-instruct/2"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(model_name_or_path, local_files_only=True)
tokenizer.pad_token_id = tokenizer.eos_token_id
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForSequenceClassification.from_pretrained(
    model_name_or_path,
    quantization_config=quantization_config,
    device_map="auto",
    num_labels=2,
    local_files_only=True,
    trust_remote_code=True,
    # ignore_mismatched_sizes=True,
)

model.gradient_checkpointing_enable()
model.enable_input_require_grads()
model.config.use_cache = False
model.config.pad_token_id = tokenizer.pad_token_id
model.config.pretraining_tp = 1

# Optionally adjust model dropout
# model.config.hidden_dropout_prob = 0.1
# model.config.attention_dropout = 0.1

# Configure LoRA for Fine‑Tuning
lora_config = LoraConfig(
    r=128,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    bias="none",
    use_rslora=True,
    task_type="SEQ_CLS",
)
model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, lora_config)
print("LoRA-enabled model. Trainable parameters:")
model.print_trainable_parameters()

# Create Custom Dataset for Sequence Classification

In [ ]:
class SentimentDataset(Dataset):
    def __init__(self, df, tokenizer, max_length=512, is_train=True, include_true_label=False):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.is_train = is_train
        self.include_true_label = include_true_label

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        sentence = row["sentence"].strip()
        language = row["language"].strip()
        # Build a simple classification prompt.
        text = f"Classify the sentiment of the following text in {language}:\n\"{sentence}\""
        tokenized = self.tokenizer(
            text,
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt",
        )
        input_ids = tokenized.input_ids.squeeze(0)
        attention_mask = tokenized.attention_mask.squeeze(0)

        item = {
            "input_ids": input_ids,
            "attention_mask": attention_mask
        }
        # For training/evaluation, convert label string to integer (0: Negative, 1: Positive)
        if self.is_train and "label" in row:
            label_str = row["label"].strip()
            label_int = 1 if label_str.lower() == "positive" else 0
            item["labels"] = label_int
            if self.include_true_label:
                item["true_label"] = label_int
        # For inference, pass along the sample ID.
        if not self.is_train and "ID" in row:
            item["ID"] = row["ID"]
        return item

# Data Collator for Dynamic Padding

In [ ]:
def collate_fn(batch):
    input_ids = [item["input_ids"] for item in batch]
    attention_mask = [item["attention_mask"] for item in batch]
    padded = tokenizer.pad(
        {"input_ids": input_ids, "attention_mask": attention_mask},
        return_tensors="pt"
    )
    if "labels" in batch[0]:
        labels = [item["labels"] for item in batch]
        padded["labels"] = torch.tensor(labels, dtype=torch.long)
    if "ID" in batch[0]:
        padded["ID"] = [item["ID"] for item in batch]
    return padded

# Load and Prepare Data

In [ ]:
train_df = pd.read_csv("/kaggle/input/multi-lingual-sentiment-analysis/train.csv")
test_df = pd.read_csv("/kaggle/input/multi-lingual-sentiment-analysis/test.csv")

train_dataset = SentimentDataset(train_df, tokenizer, max_length=512, is_train=True, include_true_label=False)
eval_dataset = SentimentDataset(
    train_df.sample(frac=0.1, random_state=42).reset_index(drop=True),
    tokenizer, max_length=512, is_train=True, include_true_label=True
)
test_dataset = SentimentDataset(test_df, tokenizer, max_length=512, is_train=False)

# Setting Training Arguments

In [ ]:
training_args = TrainingArguments(
    output_dir="./qlora_finetuned",
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=3,
    learning_rate=2e-5,
    warmup_ratio=0.03,
    max_grad_norm=0.3,
    optim="paged_adamw_32bit",
    weight_decay=0.01,
    logging_steps=10,
    save_strategy="steps",
    save_steps=100,
    fp16=True,
    gradient_accumulation_steps=8,
    save_total_limit=2,
    eval_strategy="steps",
    eval_steps=10,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    lr_scheduler_type="cosine",
    report_to="none",
)

# Compute Metrics for Sequence Classification

In [ ]:
def compute_metrics(eval_preds):
    logits, labels = eval_preds
    preds = logits.argmax(axis=1)
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average="weighted")
    report = classification_report(labels, preds, output_dict=True)
    cm = confusion_matrix(labels, preds)
    return {"accuracy": acc, "f1": f1, "classification_report": report, "confusion_matrix": cm}

# Instantiate Trainer with Early Stopping and Checkpoint Resumption

In [ ]:
if not os.path.exists(training_args.output_dir):
    os.makedirs(training_args.output_dir, exist_ok=True)
last_checkpoint = get_last_checkpoint(training_args.output_dir)

callbacks = [EarlyStoppingCallback(early_stopping_patience=3)]

if last_checkpoint is None:
    print("No checkpoint found. Training from scratch.")
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        data_collator=collate_fn,
        callbacks=callbacks,
        compute_metrics=compute_metrics,
    )
    trainer.train()
else:
    print(f"Resuming training from checkpoint {last_checkpoint}")
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        data_collator=collate_fn,
        callbacks=callbacks,
        compute_metrics=compute_metrics,
    )
    trainer.train(resume_from_checkpoint=last_checkpoint)

# Save the fine‑tuned model and tokenizer after training.

In [ ]:
model.save_pretrained("./multi_lingual_seq_cls_model")
tokenizer.save_pretrained("./multi_lingual_seq_cls_model")

# 10. Inference on Test Set & Generate Submission
test_dataloader = DataLoader(test_dataset, batch_size=4, collate_fn=collate_fn)
model.eval()
predictions = []
sample_ids = []
label_map = {0: "Negative", 1: "Positive"}

with torch.no_grad():
    for batch in test_dataloader:
        input_ids = batch["input_ids"].to(model.device)
        attention_mask = batch["attention_mask"].to(model.device)
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits
        preds = torch.argmax(logits, dim=1)
        predictions.extend(preds.cpu().numpy().tolist())
        sample_ids.extend(batch["ID"])
predicted_labels = [label_map[pred] for pred in predictions]

# Save Submission

In [ ]:
submission_df = pd.DataFrame({"ID": sample_ids, "label": predicted_labels}).sort_values("ID")
submission_df.to_csv("/kaggle/working/submission.csv", index=False)
print("Submission file saved as /kaggle/working/submission.csv")